In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cuda


In [4]:
def find_project_root():

    current = Path.cwd().resolve()

    while current != current.parent:

        if (current / ".git").exists():
            return current

        current = current.parent

    raise FileNotFoundError(
        "Could not find project root."
    )


PROJECT_ROOT = find_project_root()

DATA_ROOT = (
    PROJECT_ROOT
    / "data"
    / "GTSRB"
)

SPLIT_DIR = (
    PROJECT_ROOT
    / "splits"
)

In [5]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

In [6]:
from Common.dataset import create_dataloaders

from Common.models import (
    ModelB,
    count_parameters
)

from Common.training import (
    train_model
)

In [7]:
train_df = pd.read_csv(
    SPLIT_DIR / "train_split.csv"
)

val_df = pd.read_csv(
    SPLIT_DIR / "val_split.csv"
)

test_df = pd.read_csv(
    SPLIT_DIR / "test_split.csv"
)

In [8]:
train_loader, val_loader, test_loader = (
    create_dataloaders(
        train_df,
        val_df,
        test_df,
        DATA_ROOT,
        batch_size=32,
        num_workers=0
    )
)

In [9]:
model = ModelB().to(device)

In [10]:
criterion = nn.CrossEntropyLoss()

In [11]:
optimizer_configs = {

    "Adam": lambda parameters:
        optim.Adam(
            parameters,
            lr=1e-3
        ),

    "SGD": lambda parameters:
        optim.SGD(
            parameters,
            lr=1e-2
        ),

    "SGD_Momentum": lambda parameters:
        optim.SGD(
            parameters,
            lr=1e-2,
            momentum=0.9
        )
}

In [ ]:
optimizer_results = {}

for name, optimizer_function in optimizer_configs.items():

    print("\n" + "=" * 50)
    print(f"Training with {name}")
    print("=" * 50)

    # Reset random seeds for a fair comparison
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    # IMPORTANT:
    # Every optimizer gets a completely fresh Model B
    model = ModelB().to(device)

    optimizer = optimizer_function(
        model.parameters()
    )

    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        epochs=20,
        device=device
    )

    optimizer_results[name] = history


Training with Adam
Epoch 01/20 | Train Loss: 3.3120 | Train Acc: 0.0973 | Val Loss: 2.9928 | Val Acc: 0.1493 | Time: 60.71s
Epoch 02/20 | Train Loss: 2.6445 | Train Acc: 0.2146 | Val Loss: 2.0573 | Val Acc: 0.3567 | Time: 61.23s
Epoch 03/20 | Train Loss: 1.9650 | Train Acc: 0.3567 | Val Loss: 1.5932 | Val Acc: 0.4465 | Time: 60.02s
Epoch 04/20 | Train Loss: 1.6523 | Train Acc: 0.4400 | Val Loss: 1.3819 | Val Acc: 0.5210 | Time: 55.95s
Epoch 05/20 | Train Loss: 1.4026 | Train Acc: 0.5186 | Val Loss: 1.1168 | Val Acc: 0.6290 | Time: 55.43s
Epoch 06/20 | Train Loss: 1.1146 | Train Acc: 0.6178 | Val Loss: 0.8048 | Val Acc: 0.7266 | Time: 56.56s
Epoch 07/20 | Train Loss: 0.8562 | Train Acc: 0.7035 | Val Loss: 0.5387 | Val Acc: 0.8216 | Time: 56.84s
Epoch 08/20 | Train Loss: 0.6516 | Train Acc: 0.7731 | Val Loss: 0.3680 | Val Acc: 0.8796 | Time: 56.02s
Epoch 09/20 | Train Loss: 0.5182 | Train Acc: 0.8214 | Val Loss: 0.2903 | Val Acc: 0.9053 | Time: 62.44s
Epoch 10/20 | Train Loss: 0.4090 | 

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for name, history in optimizer_results.items():

    plt.plot(
        range(1, len(history["val_accuracy"]) + 1),
        history["val_accuracy"],
        label=name
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Optimizer Comparison - Validation Accuracy")

plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

for name, history in optimizer_results.items():

    plt.plot(
        range(1, len(history["val_loss"]) + 1),
        history["val_loss"],
        label=name
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Optimizer Comparison - Validation Loss")

plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
optimizer_summary = []

for name, history in optimizer_results.items():

    best_val_acc = max(
        history["val_accuracy"]
    )

    final_val_acc = history[
        "val_accuracy"
    ][-1]

    final_val_loss = history[
        "val_loss"
    ][-1]

    avg_epoch_time = np.mean(
        history["epoch_time"]
    )

    optimizer_summary.append({
        "Optimizer": name,
        "Best Validation Accuracy": best_val_acc,
        "Final Validation Accuracy": final_val_acc,
        "Final Validation Loss": final_val_loss,
        "Average Epoch Time (s)": avg_epoch_time
    })


optimizer_df = pd.DataFrame(
    optimizer_summary
)

optimizer_df

In [ ]:
# Display optimizer comparison
print(optimizer_df)

# Create result folders if they do not already exist
RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
METRICS_DIR = RESULTS_DIR / "metrics"

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save optimizer comparison table
optimizer_df.to_csv(
    TABLES_DIR / "optimizer_comparison.csv",
    index=False
)

print(
    "Optimizer comparison saved to:",
    TABLES_DIR / "optimizer_comparison.csv"
)

Selecting Final Optimizer

In [ ]:
ranked_optimizers = optimizer_df.sort_values(
    by=[
        "Best Validation Accuracy",
        "Final Validation Loss"
    ],
    ascending=[
        False,
        True
    ]
)

print("Optimizer ranking:")
display(ranked_optimizers)

In [ ]:
selected_optimizer = ranked_optimizers.iloc[0][
    "Optimizer"
]

print(
    "Selected optimizer:",
    selected_optimizer
)

In [ ]:
selected_optimizer_path = (
    METRICS_DIR
    / "selected_optimizer.txt"
)

selected_optimizer_path.write_text(
    str(selected_optimizer)
)

print(
    "Selected optimizer saved as:",
    selected_optimizer
)

print(
    "File:",
    selected_optimizer_path
)